In [1]:
import argparse
import os
import pathlib
import sys

import numpy as np
import pandas as pd
from image_analysis_3D.featurization_utils.feature_writing_utils import (
    format_morphology_feature_name,
)
from image_analysis_3D.featurization_utils.neighbors_utils import (
    classify_cells_into_shells,
    euclidean_distance_from_centroid,
    mahalanobis_distance_from_centroid,
)
from image_analysis_3D.file_utils.arg_parsing_utils import parse_args
from image_analysis_3D.file_utils.notebook_init_utils import (
    bandicoot_check,
    init_notebook,
)
from tqdm import tqdm

root_dir, in_notebook = init_notebook()

profile_base_dir = bandicoot_check(
    pathlib.Path(os.path.expanduser("~/mnt/bandicoot")).resolve(), root_dir
)

In [2]:
if not in_notebook:
    args = parse_args()
    well_fov = args["well_fov"]
    patient = args["patient"]
    image_based_profiles_subparent_name = args["image_based_profiles_subparent_name"]

else:
    patient = "NF0014_T1"
    well_fov = "C4-1"
    image_based_profiles_subparent_name = "image_based_profiles"

In [3]:
def centroid_within_bbox_detection(
    centroid: tuple,
    bbox: tuple,
) -> bool:
    """
    Check if the centroid is within the bbox

    Parameters
    ----------
    centroid : tuple
        Centroid of the object in the order of (z, y, x)
        Order of the centroid is important
    bbox : tuple
        Where the bbox is in the order of (z_min, y_min, x_min, z_max, y_max, x_max)
        Order of the bbox is important

    Returns
    -------
    bool
        True if the centroid is within the bbox, False otherwise
    """
    z_min, y_min, x_min, z_max, y_max, x_max = bbox
    z, y, x = centroid
    # check if the centroid is within the bbox
    if (
        z >= z_min
        and z <= z_max
        and y >= y_min
        and y <= y_max
        and x >= x_min
        and x <= x_max
    ):
        return True
    else:
        return False

### Pathing

In [4]:
# input paths
sc_profile_path = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/0.converted_profiles/{well_fov}/sc_profiles_{well_fov}.parquet"
).resolve(strict=True)
organoid_profile_path = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/0.converted_profiles/{well_fov}/organoid_profiles_{well_fov}.parquet"
).resolve(strict=True)
nucleocentric_profile_path = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/0.converted_profiles/{well_fov}/nucleocentric_profiles_{well_fov}.parquet"
).resolve(strict=True)
# output paths
sc_profile_output_path = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/1.related_profiles/{well_fov}/sc_profiles_{well_fov}_related.parquet"
).resolve()
organoid_profile_output_path = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/1.related_profiles/{well_fov}/organoid_profiles_{well_fov}_related.parquet"
).resolve()
nucleocentric_profile_output_path = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/1.related_profiles/{well_fov}/nucleocentric_profiles_{well_fov}_related.parquet"
).resolve()
sc_profile_output_path.parent.mkdir(parents=True, exist_ok=True)

In [5]:
sc_profile_df = pd.read_parquet(sc_profile_path)
nucleocentric_df = pd.read_parquet(nucleocentric_profile_path)
organoid_profile_df = pd.read_parquet(organoid_profile_path)
print(f"Single-cell profile shape: {sc_profile_df.shape}")
print(f"Nucleocentric profile shape: {nucleocentric_df.shape}")
print(f"Organoid profile shape: {organoid_profile_df.shape}")

Single-cell profile shape: (55, 10011)
Nucleocentric profile shape: (55, 3074)
Organoid profile shape: (1, 3337)


In [6]:
# initialize the parent organoid column
sc_profile_df.insert(2, "ParentOrganoid", -1)

In [7]:
x_y_z_sc_colnames = [
    x
    for x in sc_profile_df.columns
    if "area" in x.lower() and "center" in x.lower() and "nuclei" in x.lower()
]
x_y_z_sc_colnames

['Nuclei_NoChannel_AreaSizeShape_CenterX',
 'Nuclei_NoChannel_AreaSizeShape_CenterY',
 'Nuclei_NoChannel_AreaSizeShape_CenterZ']

In [8]:
organoid_bbox_colnames = [
    x
    for x in organoid_profile_df.columns
    if "area" in x.lower() and ("min" in x.lower() or "max" in x.lower())
]
organoid_bbox_colnames = sorted(organoid_bbox_colnames)

In [9]:
sc_centroids = sc_profile_df[
    x_y_z_sc_colnames
].values  # alphabetically sorted to be in the order of x,y,z

In [10]:
# Initialize parent_organoid to -1
sc_profile_df["ParentOrganoid"] = -1

# Extract single-cell centroids as numpy array for faster access
sc_centroids = sc_profile_df[x_y_z_sc_colnames].values  # (N_cells, 3) array
# reshape the centroids to be in z,y,x order for easier comparison with bbox
sc_centroids = sc_centroids[:, [2, 1, 0]]  # reorder to z,y,x

# Loop through organoids with progress bar
for organoid_index, organoid_row in tqdm(
    organoid_profile_df.iterrows(),
    total=len(organoid_profile_df),
    desc="Assigning cells to organoids",
):
    # Get organoid bbox
    organoid_bbox = (
        organoid_row[organoid_bbox_colnames[5]],  # z_min
        organoid_row[organoid_bbox_colnames[4]],  # y_min
        organoid_row[organoid_bbox_colnames[3]],  # x_min
        organoid_row[organoid_bbox_colnames[2]],  # z_max
        organoid_row[organoid_bbox_colnames[1]],  # y_max
        organoid_row[organoid_bbox_colnames[0]],  # x_max
    )

    z_min, y_min, x_min, z_max, y_max, x_max = organoid_bbox

    # Vectorized bbox check - much faster!
    mask = (
        (sc_centroids[:, 0] >= z_min)  # z
        & (sc_centroids[:, 0] <= z_max)  # z
        & (sc_centroids[:, 1] >= y_min)
        & (sc_centroids[:, 1] <= y_max)
        & (sc_centroids[:, 2] >= x_min)
        & (sc_centroids[:, 2] <= x_max)
    )

    # Only assign if cell doesn't already have a parent
    unassigned_mask = sc_profile_df["ParentOrganoid"] == -1
    final_mask = mask & unassigned_mask

    # Assign parent organoid to matching cells
    sc_profile_df.loc[final_mask, "ParentOrganoid"] = organoid_row["object_id"]

print(f"Assigned {(sc_profile_df['ParentOrganoid'] != -1).sum()} cells to organoids")
print(f"Unassigned cells: {(sc_profile_df['ParentOrganoid'] == -1).sum()}")

Assigning cells to organoids: 100%|██████████| 1/1 [00:00<00:00, 583.27it/s]

Assigned 55 cells to organoids
Unassigned cells: 0


### Add single-cell counts for each organoid

In [11]:
organoid_sc_counts = (
    sc_profile_df["ParentOrganoid"]
    .value_counts()
    .to_frame(name="OrganoidSingleCellCount")
    .reset_index()
)
# merge the organoid profile with the single-cell counts
organoid_profile_df = pd.merge(
    organoid_profile_df,
    organoid_sc_counts,
    left_on="object_id",
    right_on="ParentOrganoid",
    how="left",
).drop(columns=["ParentOrganoid"])
sc_count = organoid_profile_df.pop("OrganoidSingleCellCount")
organoid_profile_df.insert(2, "OrganoidSingleCellCount", sc_count)

Even if the file is empty we still want to add it to the final dataframe dictionary so that we can merge on the same columns later.
This will help with file-based checking and merging.


In [12]:
# replace NaN with 0 for organoids that have no assigned cells
organoid_profile_df["OrganoidSingleCellCount"] = (
    organoid_profile_df["OrganoidSingleCellCount"].fillna(0).astype(int)
)
organoid_profile_df.head()

,object_id,image_set,OrganoidSingleCellCount,Organoid_NoChannel_AreaSizeShape_Volume,Organoid_NoChannel_AreaSizeShape_CenterX,Organoid_NoChannel_AreaSizeShape_CenterY,Organoid_NoChannel_AreaSizeShape_CenterZ,Organoid_NoChannel_AreaSizeShape_BboxVolume,Organoid_NoChannel_AreaSizeShape_MinX,Organoid_NoChannel_AreaSizeShape_MaxX,...,Organoid_Mito_Texture_DifferenceEntropy-256-3,Organoid_Mito_Texture_DifferenceVariance-256-3,Organoid_Mito_Texture_Entropy-256-3,Organoid_Mito_Texture_InformationMeasureOfCorrelation1-256-3,Organoid_Mito_Texture_InformationMeasureOfCorrelation2-256-3,Organoid_Mito_Texture_InverseDifferenceMoment-256-3,Organoid_Mito_Texture_SumAverage-256-3,Organoid_Mito_Texture_SumEntropy-256-3,Organoid_Mito_Texture_SumVariance-256-3,Organoid_Mito_Texture_Variance-256-3
0,1,C4-1,55,27421038.0,711.460877,936.634612,19.864556,50391450.0,267,1082,...,3.316532,0.000513,6.680345,-0.057098,0.564855,0.228559,71.61064,4.750468,104.779099,35.706212


In [13]:
if organoid_profile_df.empty:
    # add a row with 0 values
    organoid_profile_df.loc[len(organoid_profile_df)] = [0] * len(
        organoid_profile_df.columns
    )
    organoid_profile_df["image_set"] = well_fov

In [14]:
print(f"Single-cell profile shape: {sc_profile_df.shape}")

Single-cell profile shape: (55, 10012)


In [15]:
if sc_profile_df.empty:
    # add a row with Na values
    sc_profile_df.loc[len(sc_profile_df)] = [None] * len(sc_profile_df.columns)
    sc_profile_df["image_set"] = well_fov

In [16]:
# add the parent organoid to nucleocentric features
nucleocentric_df = pd.merge(
    nucleocentric_df,
    sc_profile_df[["object_id", "image_set", "ParentOrganoid"]],
    on=["object_id", "image_set"],
    how="left",
)

## Get single cell and organoid relationships and spatial distributions

In [17]:
x_y_z_organoid_centroid_colnames = [
    x
    for x in organoid_profile_df.columns
    if "area" in x.lower() and "center" in x.lower()
]
x_y_z_organoid_bbox_colnames = [
    x
    for x in organoid_profile_df.columns
    if "area" in x.lower() and ("min" in x.lower() or "max" in x.lower())
]

In [18]:
results = []

# get the organoid id and the single-cells for each

organoid_ids = organoid_profile_df["object_id"]

# organoid_id = organoid_ids[0]

for organoid_id in organoid_ids:
    organoid_centroid = (
        organoid_profile_df.loc[
            organoid_profile_df["object_id"] == organoid_id,
            x_y_z_organoid_centroid_colnames,
        ]
        .apply(pd.to_numeric, errors="coerce")
        .iloc[0]
        .to_numpy(dtype=float)
    )
    organoid_bbox = organoid_profile_df.loc[
        organoid_profile_df["object_id"] == organoid_id, x_y_z_organoid_bbox_colnames
    ].values[0]
    single_cells_in_organoid = sc_profile_df[
        sc_profile_df["ParentOrganoid"] == organoid_id
    ]
    if single_cells_in_organoid.empty:
        print(f"No single cells assigned to organoid {organoid_id}")
        continue

    single_cells_centroids = (
        single_cells_in_organoid[x_y_z_sc_colnames]
        .apply(pd.to_numeric, errors="coerce")
        .to_numpy(dtype=float)
    )

    valid_rows = ~pd.isna(single_cells_centroids).any(axis=1)
    single_cells_centroids = single_cells_centroids[valid_rows]
    single_cells_in_organoid = single_cells_in_organoid.loc[valid_rows]

    if single_cells_centroids.shape[0] == 0:
        continue

    # convert to a dict with the key being the object_id
    # rename the centroids to z,y.x
    single_cells_centroids_dict = {
        "object_id": single_cells_in_organoid["object_id"].to_numpy(),
        "z": single_cells_in_organoid[x_y_z_sc_colnames[2]].to_numpy(dtype=float),
        "y": single_cells_in_organoid[x_y_z_sc_colnames[1]].to_numpy(dtype=float),
        "x": single_cells_in_organoid[x_y_z_sc_colnames[0]].to_numpy(dtype=float),
    }

    euclidean_distance = euclidean_distance_from_centroid(
        single_cells_centroids, organoid_centroid
    )
    mahalanobis_distance = mahalanobis_distance_from_centroid(
        single_cells_centroids, organoid_centroid
    )
    shell_classification, centroid = classify_cells_into_shells(
        coords=single_cells_centroids_dict,
        n_shells=4,
        method="mahalanobis",
        min_cells_per_shell=3,
        centroid=organoid_centroid,
    )

    shell_classification_df = pd.DataFrame(shell_classification)
    shell_classification_df["ParentOrganoid"] = organoid_id
    results.append(shell_classification_df)

In [19]:
if results:
    df = pd.concat([pd.DataFrame(r) for r in results], ignore_index=True)

else:
    df = pd.DataFrame(columns=["object_id", "ParentOrganoid"])

# rename the columns

df.rename(
    columns={
        col: format_morphology_feature_name(
            compartment="Nuclei",
            feature_type="Neighbors",
            channel="NoChannel",
            measurement=col,
        )
        for col in df.columns
        if col not in ["object_id", "ParentOrganoid"]
    },
    inplace=True,
)

In [20]:
# concat the shell classification with the single cell profile df to get the full single cell profile with the shell classification and the parent organoid id
sc_profile_with_shells_df = pd.merge(
    sc_profile_df,
    df,
    left_on=["object_id", "ParentOrganoid"],
    right_on=["object_id", "ParentOrganoid"],
    how="left",
)

### Save the profiles

In [21]:
organoid_profile_df.to_parquet(organoid_profile_output_path, index=False)
organoid_profile_df.head()

,object_id,image_set,OrganoidSingleCellCount,Organoid_NoChannel_AreaSizeShape_Volume,Organoid_NoChannel_AreaSizeShape_CenterX,Organoid_NoChannel_AreaSizeShape_CenterY,Organoid_NoChannel_AreaSizeShape_CenterZ,Organoid_NoChannel_AreaSizeShape_BboxVolume,Organoid_NoChannel_AreaSizeShape_MinX,Organoid_NoChannel_AreaSizeShape_MaxX,...,Organoid_Mito_Texture_DifferenceEntropy-256-3,Organoid_Mito_Texture_DifferenceVariance-256-3,Organoid_Mito_Texture_Entropy-256-3,Organoid_Mito_Texture_InformationMeasureOfCorrelation1-256-3,Organoid_Mito_Texture_InformationMeasureOfCorrelation2-256-3,Organoid_Mito_Texture_InverseDifferenceMoment-256-3,Organoid_Mito_Texture_SumAverage-256-3,Organoid_Mito_Texture_SumEntropy-256-3,Organoid_Mito_Texture_SumVariance-256-3,Organoid_Mito_Texture_Variance-256-3
0,1,C4-1,55,27421038.0,711.460877,936.634612,19.864556,50391450.0,267,1082,...,3.316532,0.000513,6.680345,-0.057098,0.564855,0.228559,71.61064,4.750468,104.779099,35.706212


In [22]:
sc_profile_with_shells_df.to_parquet(sc_profile_output_path, index=False)
sc_profile_with_shells_df.head()

,object_id,image_set,ParentOrganoid,Nuclei_NoChannel_AreaSizeShape_Volume,Nuclei_NoChannel_AreaSizeShape_CenterX,Nuclei_NoChannel_AreaSizeShape_CenterY,Nuclei_NoChannel_AreaSizeShape_CenterZ,Nuclei_NoChannel_AreaSizeShape_BboxVolume,Nuclei_NoChannel_AreaSizeShape_MinX,Nuclei_NoChannel_AreaSizeShape_MaxX,...,Cytoplasm_ER_Texture_InverseDifferenceMoment-256-3,Cytoplasm_ER_Texture_SumAverage-256-3,Cytoplasm_ER_Texture_SumEntropy-256-3,Cytoplasm_ER_Texture_SumVariance-256-3,Cytoplasm_ER_Texture_Variance-256-3,Nuclei_NoChannel_Neighbors_ShellAssignments,Nuclei_NoChannel_Neighbors_DistancesFromCenter,Nuclei_NoChannel_Neighbors_DistancesFromExterior,Nuclei_NoChannel_Neighbors_NormalizedDistancesFromCenter,Nuclei_NoChannel_Neighbors_ShellsUsed
0,1,C4-1,1,7480.0,881.128610,443.802139,1.488770,10200.0,857,907,...,NaN,NaN,NaN,NaN,NaN,2,2.115407,1.241413,0.630182,4
1,2,C4-1,1,37965.0,570.619781,889.462874,4.402107,53088.0,533,612,...,NaN,NaN,NaN,NaN,NaN,1,1.624583,1.732237,0.483965,4
2,3,C4-1,1,45736.0,630.840716,980.077226,4.681083,80442.0,576,685,...,NaN,NaN,NaN,NaN,NaN,1,1.398383,1.958437,0.416580,4
3,4,C4-1,1,32804.0,513.994940,1233.819229,3.297586,55692.0,471,562,...,NaN,NaN,NaN,NaN,NaN,2,2.027695,1.329125,0.604052,4
4,5,C4-1,1,78836.0,805.882478,657.815478,9.888135,139956.0,752,859,...,NaN,NaN,NaN,NaN,NaN,1,1.170524,2.186296,0.348700,4


In [23]:
nucleocentric_df.to_parquet(nucleocentric_profile_output_path, index=False)
nucleocentric_df.head()

,object_id,image_set,Nucleocentric_ER_CHAMMI75_Feature0,Nucleocentric_ER_CHAMMI75_Feature1,Nucleocentric_ER_CHAMMI75_Feature10,Nucleocentric_ER_CHAMMI75_Feature100,Nucleocentric_ER_CHAMMI75_Feature101,Nucleocentric_ER_CHAMMI75_Feature102,Nucleocentric_ER_CHAMMI75_Feature103,Nucleocentric_ER_CHAMMI75_Feature104,...,Nucleocentric_DNA_SAMMed3D_Feature91,Nucleocentric_DNA_SAMMed3D_Feature92,Nucleocentric_DNA_SAMMed3D_Feature93,Nucleocentric_DNA_SAMMed3D_Feature94,Nucleocentric_DNA_SAMMed3D_Feature95,Nucleocentric_DNA_SAMMed3D_Feature96,Nucleocentric_DNA_SAMMed3D_Feature97,Nucleocentric_DNA_SAMMed3D_Feature98,Nucleocentric_DNA_SAMMed3D_Feature99,ParentOrganoid
0,1,C4-1,-2.034619,-3.223986,5.187386,0.839193,-4.609393,-1.314024,2.809356,0.652013,...,-0.050433,0.088730,-0.010508,0.028697,0.023633,-0.013850,0.238845,0.333358,0.229721,1
1,2,C4-1,-2.077653,-3.769540,7.392309,-2.904451,0.304836,-0.125569,3.759428,1.653862,...,-0.063565,0.012365,-0.010451,0.018819,-0.035347,-0.072271,0.225065,0.375739,0.230366,1
2,3,C4-1,1.937732,-3.106618,3.096840,2.169843,-0.647295,-3.975330,2.363535,-1.013884,...,-0.074491,-0.133589,-0.010789,0.014343,-0.093814,0.208879,0.276905,0.316322,0.034441,1
3,4,C4-1,0.399118,-2.973527,2.263089,2.779028,0.539425,-0.112796,4.970295,-2.148762,...,-0.088927,0.062739,-0.010761,0.023554,0.038025,0.025290,0.201641,0.315303,0.227351,1
4,5,C4-1,-2.808134,-5.294312,0.994752,2.960990,-2.145890,1.459159,6.289212,-0.280198,...,-0.036514,0.055145,-0.010306,0.020019,-0.006451,-0.026692,0.222935,0.391211,0.207672,1
